# Real Machine Learning for NIDS
Welcome! Here we will build a REAL Machine Learning algorithm without mocked outputs. We'll use a real open-source cybersecurity dataset (`KDD Cup 99`) to train a Random Forest.

## 1. Import Libraries
We use `pandas` to structure our tabular data, and `scikit-learn` to fetch real cybersecurity data and train our Machine Learning Model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

## 2. Load Real Network Dataset (KDD Cup 99)
We are going to download 10% of the KDD Cup 99 dataset directly from the scikit-learn dataset hub. This contains hundreds of thousands of real recorded network connections.

In [ ]:
print("Downloading KDD Cup 99 dataset (this might take a few seconds)...")
kdd = fetch_kddcup99(as_frame=True, percent10=True)
df = kdd.frame

# Output target label is called 'labels'. Anything that isn't 'normal.' is an attack.
df['is_anomaly'] = (df['labels'] != b'normal.').astype(int)

print(f"Loaded dataset with {len(df)} connections.")
print("Distribution of Normal (0) vs Attack (1):")
print(df['is_anomaly'].value_counts())

# Display first few rows
display(df.head())

## 3. Preprocess Data
The dataset contains categorical strings (like 'tcp', 'udp', 'http') which Machine Learning models can't understand. We must decode the bytes and One-Hot Encode them into numerical 0s and 1s.

In [ ]:
print("Starting Preprocessing...")
# The KDD dataset uses byte-strings (b'tcp'). We need to decode them to normal text.
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Drop the original 'labels' column because we have our binary 'is_anomaly' target
df = df.drop('labels', axis=1)

# We will use a smaller subset of columns to make training fast and avoid complexity for now
features_to_use = ['duration', 'protocol_type', 'service', 'src_bytes', 'dst_bytes', 'count', 'is_anomaly']
df_subset = df[features_to_use]

# Convert string columns (protocol_type, service) into discrete columns (One-Hot Encoding)
df_encoded = pd.get_dummies(df_subset, columns=['protocol_type', 'service'], drop_first=True)

# Split Features (X) and Labels (y)
X = df_encoded.drop('is_anomaly', axis=1)
y = df_encoded['is_anomaly']

print("Splitting into Train/Test subsets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. Train the Random Forest Model
Now we plug the data into our real machine learning algorithm.

In [ ]:
model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

print("Training the machine learning model on real historical network patterns...")
model.fit(X_train, y_train)
print("Training complete!")

## 5. Evaluate
Let's see how our real model performs on the test set it has never seen.

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, predictions, target_names=['Normal', 'Attack']))

## 6. Save Model & Training Columns
We need to export both the `model` AND the structure of `X_train.columns` so our FastAPI Python engine maps packets perfectly.

In [ ]:
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

# Export Model
model_path = os.path.join(models_dir, "nids_real_rf_model.pkl")
joblib.dump(model, model_path)
print(f"Saved AI Model to {model_path}")

# Export the columns so the actual live traffic knows what format to take
columns_path = os.path.join(models_dir, "nids_features.pkl")
joblib.dump(X_train.columns.tolist(), columns_path)
print(f"Saved Feature map to {columns_path}")